

## Step 1: Data Preparation and Chunking
We need to load the CSV file and break the long articles into smaller, manageable pieces (called "chunks"). This is crucial because we have a limitation of maximum 1024 tokens per chunk, and we want to avoid processing the entire dataset at once to stay within the 5 USD budget constraint.  

In [6]:
# 1. Install the tokenizer library (run this in a new cell)
# %pip install tiktoken

import tiktoken
import pandas as pd

# 2. Initialize the tokenizer specifically for our embedding model
# This ensures our token count matches exactly what the API will calculate
tokenizer = tiktoken.encoding_for_model("text-embedding-3-small")

def create_token_chunks(text, max_tokens, overlap_tokens):
    """
    Splits text into chunks based on exact token counts, preserving context overlap.
    
    Args:
        text (str): The full article text.
        max_tokens (int): Maximum token size of each chunk.
        overlap_tokens (int): Number of tokens overlapping with the previous chunk.
        
    Returns:
        list: A list of string chunks.
    """
    # Encode the entire text string into a list of integer token IDs
    tokens = tokenizer.encode(str(text))
    chunks = []
    
    if len(tokens) == 0:
        return chunks
        
    # Iterate through the token array and create overlapping slices
    for i in range(0, len(tokens), max_tokens - overlap_tokens):
        # Extract the specific slice of tokens
        token_slice = tokens[i:i + max_tokens]
        
        # Decode the integer tokens back into a readable text string
        chunk_text = tokenizer.decode(token_slice)
        chunks.append(chunk_text)
        
    return chunks

# 3. Test the new, precise token-based chunker
CHUNK_SIZE_TOKENS = 500
OVERLAP_TOKENS = 100

df = pd.read_csv("medium-english-50mb.csv")
sample_text = df.iloc[0]['text']

# Apply the new function
token_chunks = create_token_chunks(sample_text, CHUNK_SIZE_TOKENS, OVERLAP_TOKENS)

print(f"Total precise tokens in article: {len(tokenizer.encode(str(sample_text)))}")
print(f"Created {len(token_chunks)} precise token-based chunks.")

Total precise tokens in article: 1073
Created 3 precise token-based chunks.


### Step 2: Pinecone Vector Database Setup  
We initialize the Pinecone vector database to act as our "library" for storing article embeddings. We define an index with 1536 dimensions to match the ZYRANGG-text-embedding-3-small model and ensure the index is active for querying

In [ ]:
from pinecone import Pinecone, ServerlessSpec


PINECONE_API_KEY = "****"

# Connect to the service
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medium-rag"

# Check if the index exists, and if not - create it
if index_name not in pc.list_indexes().names():
    print("Creating index...")
    pc.create_index(
        name=index_name,
        dimension=1536, # Critical: Must match the text-embedding-3-small model
        metric="cosine", # Standard metric for measuring distance between texts
        spec=ServerlessSpec(
            cloud='aws', 
            region='us-east-1' # Common free tier region
        )
    )

# Connect to our specific index
index = pc.Index(index_name)
print("Successfully connected to Pinecone index!")

Successfully connected to Pinecone index!


### Step 3: Precise Tokenization with Tiktoken  
To strictly adhere to the 1024-token limit and optimize embedding costs, we use the tiktoken library to perform sub-word tokenization. This ensures our chunking logic aligns perfectly with the embedding model's input requirements rather than relying on word count estimates.

In [ ]:
import pandas as pd
from openai import OpenAI
import tiktoken
from pinecone import Pinecone

# --- Configuration ---
# Replace these with your actual platform keys
API_KEY = ""
PINECONE_API_KEY = ""
INDEX_NAME = "medium-rag"
EMBEDDING_MODEL = "NBUECSE-text-embedding-3-small"
BATCH_SIZE = 100  # Process 100 chunks per API call to optimize performance

# Initialize API Clients
# Initialize the client with the custom base URL to route requests correctly
llm_client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.llmod.ai/v1" # Verify this exact URL format in your course documentation
)
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)

# --- Batch Processing Logic ---
def process_and_upsert_batches(dataframe, max_tokens, overlap_tokens):
    """
    Iterates through the dataframe, generates token chunks, 
    creates vectors in batches, and upserts them to Pinecone.
    """
    batch_records = []
    
    for idx, row in dataframe.iterrows():
        # Treat the DataFrame index as a unique article ID
        article_id = str(idx) 
        title = row['title']
        text = row['text']
        
        # 1. Slice the text using the token chunker defined previously
        chunks = create_token_chunks(text, max_tokens, overlap_tokens)
        
        # 2. Prepare payload items for the current batch
        for chunk_idx, chunk_text in enumerate(chunks):
            # Pinecone requires a strictly unique ID for every vector
            vector_id = f"art_{article_id}_chunk_{chunk_idx}"
            
            # Append metadata and identifiers to the temporary batch list
            batch_records.append({
                "id": vector_id,
                "text": chunk_text,
                "title": title,
                "article_id": article_id
            })
            
            # 3. Trigger API call and Upsert when batch capacity is reached
            if len(batch_records) >= BATCH_SIZE:
                upsert_batch_to_pinecone(batch_records)
                batch_records = [] # Reset the batch array
                
    # 4. Flush and upsert any remaining chunks in the final partial batch
    if len(batch_records) > 0:
        upsert_batch_to_pinecone(batch_records)
        
    print("Finished processing and upserting all batches!")

def upsert_batch_to_pinecone(batch_data):
    """
    Calls the embedding API once for a list of chunks, 
    formats the output, and upserts to the vector database.
    """
    # Isolate the text strings required for the embedding model input
    texts_to_embed = [item["text"] for item in batch_data]
    
    # Generate embeddings for the entire batch in a single request
    response = llm_client.embeddings.create(
        input=texts_to_embed,
        model=EMBEDDING_MODEL
    )
    
    # Prepare the formatted payload required by Pinecone's upsert method
    pinecone_payload = []
    for i, item in enumerate(batch_data):
        # Extract the numerical vector array from the API response object
        vector_values = response.data[i].embedding
        
        # Construct the final Pinecone record containing vector and metadata
        record = {
            "id": item["id"],
            "values": vector_values,
            "metadata": {
                "text": item["text"],
                "title": item["title"],
                "article_id": item["article_id"]
            }
        }
        pinecone_payload.append(record)
        
    # Upsert the constructed payload directly into the active index
    index.upsert(vectors=pinecone_payload)
    print(f"Successfully upserted a batch of {len(pinecone_payload)} vectors.")

# --- Execution ---
# Assumes 'df' is loaded (e.g., first 50 rows) and 'create_token_chunks' exists.
print("Starting the batch embedding and upsert pipeline...")
process_and_upsert_batches(df, max_tokens=500, overlap_tokens=100)

Starting the batch embedding and upsert pipeline...
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batch of 100 vectors.
Successfully upserted a batc

### Step 4: Retrieval Logic
This step implements the system's "search engine." It converts the user's natural language question into a vector embedding and queries the Pinecone index for the top-$k$ most semantically similar article chunks. Finally, it parses the metadata to prepare the retrieved content for the LLM.  

In [10]:
# --- Step 3: Retrieval Logic ---

def retrieve_context(query, top_k=3):
    """
    Embeds the user query and searches Pinecone for the most relevant article chunks.
    
    Args:
        query (str): The natural language question from the user.
        top_k (int): The number of chunks to retrieve.
        
    Returns:
        list: A list of dictionaries containing the retrieved chunks and their metadata.
    """
    # 1. Generate the embedding vector for the user query
    query_response = llm_client.embeddings.create(
        input=[query],
        model=EMBEDDING_MODEL
    )
    query_vector = query_response.data[0].embedding
    
    # 2. Query the Pinecone index
    # include_metadata=True is required to retrieve the actual text payload
    search_results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    
    # 3. Parse and format the results to match the assignment's JSON schema requirements
    retrieved_chunks = []
    for match in search_results['matches']:
        chunk_data = {
            "article_id": match['metadata']['article_id'],
            "title": match['metadata']['title'],
            "chunk": match['metadata']['text'],
            "score": match['score']
        }
        retrieved_chunks.append(chunk_data)
        
    return retrieved_chunks

# --- Execution & Testing ---
test_query = "What is the best way to build a sustainable habit?"
results = retrieve_context(test_query, top_k=3)

print(f"Top results for: '{test_query}'\n")
for i, res in enumerate(results):
    print(f"Result {i+1} (Score: {res['score']:.4f})")
    print(f"Title: {res['title']}")
    print(f"Excerpt: {res['chunk'][:150]}...\n")

Top results for: 'What is the best way to build a sustainable habit?'

Result 1 (Score: 0.6495)
Title: Atomic Habits by James Clear
Excerpt: An easy and proven way to build good habits and break bad ones

Why are habits important?

You might not believe that small tiny habits can have a sig...

Result 2 (Score: 0.6115)
Title: Atomic Habits by James Clear
Excerpt:  old ones that you already do. This will make the new habit very easy to start and complete. For example, if you want to drink more water, make it a h...

Result 3 (Score: 0.6061)
Title: Building and Breaking Habits — An Essential Guide
Excerpt: Building and Breaking Habits — An Essential Guide

“Depending on what they are, our habits will either make or break us. We become what we repeatedly ...



### Step 5: LLM Generation and Prompt Engineering
Here, we synthesize the final answer. We format the retrieved chunks into a context-rich prompt, inject the mandatory system instructions , and call the ZYRANGG-gpt-5-mini model to generate a grounded response that strictly adheres to the provided dataset

In [11]:
# --- Step 4: LLM Generation ---

CHAT_MODEL = "NBUECSE-gpt-5-mini"

def generate_answer(query, retrieved_context):
    """
    Constructs the final prompt and generates an answer using the LLM.
    
    Args:
        query (str): The original user question.
        retrieved_context (list): The list of dictionaries containing chunks and metadata.
        
    Returns:
        tuple: (The generated answer string, The full prompt sent to the model)
    """
    
    # 1. Format the retrieved chunks into a single readable string for the LLM
    context_text = ""
    for idx, item in enumerate(retrieved_context):
        context_text += f"--- Article {idx + 1}: {item['title']} ---\n"
        context_text += f"{item['chunk']}\n\n"

    # 2. Define the strict System Prompt required by the assignment
    system_prompt = (
        "You are a Medium-article assistant that answers questions strictly and only "
        "based on the Medium articles dataset context provided to you (metadata "
        "and article passages).\n"
        "You must not use any external knowledge, the open "
        "internet, or information that is not explicitly contained in the retrieved "
        "context. If the answer cannot be determined from the provided context, "
        "respond: \"I don't know based on the provided Medium articles data.\"\n"
        "Always explain your answer using the given context, quoting or "
        "paraphrasing the relevant article passage or metadata when helpful."
    )

    # 3. Construct the user message combining the actual question and the formatted context
    user_message = f"Context:\n{context_text}\nQuestion: {query}"

    # 4. Call the Chat Completion API (Temperature parameter removed to prevent 400 error)
    response = llm_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    
    final_answer = response.choices[0].message.content
    
    # We return the full prompt setup as well, since the final API endpoint requires it
    full_prompt_payload = {
        "system": system_prompt,
        "user": user_message
    }
    
    return final_answer, full_prompt_payload

# --- Execution & Testing ---
# Using the context we retrieved in the previous step
answer, prompt_used = generate_answer(test_query, results)

print("--- Generated Answer ---")
print(answer)

--- Generated Answer ---
I don't know based on the provided Medium articles data.

Explanation: The provided context does not contain a single explicit sentence that answers "the best way" in those exact words. Based only on the given article passages, I can summarize the consistent advice they give (but not claim it as a definitive single “best” method):

- Start extremely small and make the habit easy — “Make it easy — You will never stick to your habit if it’s not easy to complete it” (Article 1) and “Start Small… it’s important to approach it in small steps” (Article 3).  
- Stack the new habit onto an existing routine — “Habit stacking — …make it a habit to drink water every time you go in your kitchen to eat” (Article 1).  
- Change your environment to reduce friction and give each space a purpose — “Change your environment — It’s so important to create an environment for yourself to start your habit and eliminate bad ones” (Article 2).  
- Repeat consistently and push through th